# ML-09 — Validation Audit (Paper Practice → Model Audit)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

First half: pick two findings from this week's research paper and practice the methodology-review question I would want a reviewer to ask me — concrete, constructive, answerable with evidence (not "bad paper"). Second half: turn the same lens on my Week-5 model. Re-run it under a grouped split, report the before/after vs a naïve random split as a "how much memorization" finding; audit features for all three leakage types; rewrite any claims that over-stretch the evidence.

Sections in order:

1. **Two paper findings + my methodology questions.**
2. **My model under an honest split (before / after).** Same model, same features, same metric — one run with a random train/test split, one run with GroupShuffleSplit by client_id. Report the GAP.
3. **Leakage audit (attack my own model).** All three taxons from the skill: label-derived features (train-with / train-without test), future/overlapping-window timeline check, and decision-derived (product flag) presence check.
4. **Claim rewrite.** Take 3 of Week-5's most confident-sounding phrases and rewrite them in "observed / measured / directional / decision-support" language.
5. Self-check.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `hunting-leakage-and-validating` + `flyrank/flyrank-data` for this task.


In [1]:
import os, sys, subprocess, importlib
import pandas as pd
import numpy as np

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

def _find_starter_csv():
    candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
        "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

STARTER_CSV = _find_starter_csv()
assert STARTER_CSV is not None, f"missing starter CSV, cwd={os.path.abspath('.')}"
_repo_root = os.path.abspath(os.path.join(STARTER_CSV, os.pardir, os.pardir, os.pardir))
OUTPUTS_DIR = os.path.join(_repo_root, "work", "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)
METRICS_JSON_PATH = os.path.join(OUTPUTS_DIR, "validation_audit.json")
print(f"Starter CSV: {STARTER_CSV}")
print(f"Repo root  : {_repo_root}")
print(f"Outputs dir: {OUTPUTS_DIR} (exists={os.path.isdir(OUTPUTS_DIR)})")
print(f"Metrics JSON: {METRICS_JSON_PATH}")

ensure_pkg("sklearn", "scikit-learn")

RAW = pd.read_csv(STARTER_CSV)
RANDOM_STATE = 42
N_FOLDS = 5
TOPK = 200
TOPK_SHORT = 50

LANE_MASK = (RAW["impressions_90d"] >= 100) & ~((RAW["avg_position"] == 0) & (RAW["impressions_90d"] < 500))
LANE = RAW[LANE_MASK].copy().reset_index(drop=True)
print(f"\nLane slice: {len(LANE):,} rows ({len(LANE)/len(RAW):.0%}), clients={LANE['client_id'].nunique()}")

LANE["severe_decline"] = (LANE["trend_pct"] < -20).astype(int)
BASE_RATE = float(LANE["severe_decline"].mean())
print(f"Label base rate severe_decline (trend_pct < -20): {BASE_RATE:.1%}")


Starter CSV: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv
Repo root  : /Users/amiroyeleke/Documents/Flyrank/flyrank-ml
Outputs dir: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs (exists=True)
Metrics JSON: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/validation_audit.json



Lane slice: 22,006 rows (73%), clients=30
Label base rate severe_decline (trend_pct < -20): 59.7%


## 1. Two paper findings + my methodology questions (constructive, answerable-with-evidence)

This exercise is not about grading the paper. It is about practicing the exact questions I want a reviewer to ask *me* — and that I should ask of any result before I trust it. Two findings from this week's session paper, and the concrete methodology question I would put on it:

---

### Paper Finding 1 (paraphrased): "Our end-to-end content-opportunity model achieves 81% precision-at-200 on the held-out test set, outperforming the FlyRank heuristic baseline by +23 percentage points."

**My methodology question (concrete, constructive):**
> **Where exactly was the held-out test set drawn from — random row split, client-held-out group split, or a future month time split? And is the reported +23 pp measured on the same rows (and the same population definition) the baseline heuristic was scored on, or was the heuristic re-evaluated on a different slice?**

Why I want to ask this: Random row split gives an inflated 23 pp "win" because the model memorizes client-tone and content-pipeline patterns; a client-held-out split is where the true 10–15 pp win lives. Also, baselines are often scored on their original export while the new model is scored on a cleaned-up test population — the gap is then partly data cleaning, not model quality. Both answers are trivially verifiable by pasting two lines of code (the split indices + baseline scorer call) in a reproduced notebook.

---

### Paper Finding 2 (paraphrased): "Feature importance shows staleness (days since last update) is the #1 driver of refresh decisions, contributing 3× as much as the next feature — hence our SHAP chart recommends prioritizing staleness in triage."

**My methodology question (concrete, constructive):**
> **Was staleness computed using days_since_last_update as a raw monotonically-increasing value, or was it binned / thresholded at exactly the same cutoffs the existing FlyRank refresh-flag product rule already uses? And were any existing-system scores or flags (e.g., refresh_flag_stale) accidentally included as features in the model? If either is true, the "#1 driver" is measuring the product rule's existing decision boundary, not a new insight about content health in the world.**

Why I want to ask this: This is the classic decision-derived-feature leakage (taxon #3 from the skill). A SHAP chart that shows "staleness is #1" when staleness was used with the same 180d cutoff that already generates `refresh_flag_stale` is not a finding about content health — it's a finding that the model memorized the old rule. The check is trivial: run once with staleness as a raw feature, once with staleness removed, and report both scores. If removing staleness collapses the 81% precision down to 57% but removes it only from features and not from the *label computation*, that number is real; if it collapses because the label itself was defined using staleness bins, that's leakage.

---

Both questions are respectfully framed, answerable by running code (not by writing paragraphs), and they target the two biggest ways "81% precision" could be worthless in practice when deployed: (1) a random-split score that doesn't hold on new clients, and (2) a feature-importance finding that just recovers the old rule. These are the exact two self-audits I will now run on my own Week-5 model in Sections 2 and 3.


## 2. My model under an honest split — before / after (random split vs client-grouped split)

The first self-audit answers the exact question I asked the paper in Section 1: *what is my score under a random (naïve) split vs an honest client-grouped split, and what is the GAP?*

Same model everywhere: Random Forest (200 trees, max_depth=6, min_samples_leaf=5, same random_state=42, same 11 features, same prec@200 / prec@50 metric). Two cross-validation regimes, 5 folds each, folds computed once and reused:

1. **BEFORE: ShuffleSplit (naïve random rows, 20% held out, 80% train).** This is the "easy" split — 80% of Client A's pages are in train, 20% in test, so the model can and will memorize Client A's blog-tone / staleness-profile / domain-language. It almost always inflates scores.
2. **AFTER: GroupShuffleSplit by client_id (honest).** 80% of clients entirely in train, 20% of clients entirely in test — per-fold overlap asserted to be empty. This mimics real deployment: the model performs for a *new client it has never seen*.

I report mean ± std across folds, and crucially — the GAP between the two numbers. The GAP is itself a finding: how much of the "model wins" claim was actually client-memorization?


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# Re-build exactly the same feature frame as w05 (11 honest features)
def build_features(frame):
    df = frame.copy()
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"].clip(lower=0).astype(float))
    df["ctr_filled"] = df["ctr"].fillna(0).clip(lower=0).astype(float)
    pos_fill = df["avg_position"].replace(0, np.nan)
    df["position_filled"] = pos_fill.fillna(99).astype(float)
    df["search_volume_filled"] = df["search_volume"].fillna(0).astype(float)
    df["log_search_volume"] = np.log1p(df["search_volume_filled"])
    df["age_days"] = df["content_age_days"].fillna(df["content_age_days"].median()).astype(float)
    df["days_since_update"] = df["days_since_last_update"].fillna(df["days_since_last_update"].median()).astype(float)
    df["staleness_bucket"] = 0
    df.loc[(df["days_since_last_update"] >= 180) & (df["days_since_last_update"] < 360), "staleness_bucket"] = 1
    df.loc[ df["days_since_last_update"] >= 360, "staleness_bucket"] = 2
    df["vis_bucket"] = 0
    df.loc[(df["impressions_90d"] >= 1_000) & (df["impressions_90d"] < 10_000), "vis_bucket"] = 1
    df.loc[ df["impressions_90d"] >= 10_000, "vis_bucket"] = 2
    df["striking_bonus"] = (
        (df["avg_position"] >= 10) & (df["avg_position"] <= 25) & (df["search_volume_filled"] >= 100)
    ).astype(int)
    df["has_word_count"] = df["word_count"].notna().astype(int)
    df["word_count_filled"] = df["word_count"].fillna(df["word_count"].median()).astype(float)
    FEATURE_COLS = [
        "log_impressions_90d", "ctr_filled", "position_filled", "log_search_volume",
        "age_days", "days_since_update", "staleness_bucket", "vis_bucket",
        "striking_bonus", "has_word_count", "word_count_filled",
    ]
    return df[FEATURE_COLS].values, FEATURE_COLS, df

X, FEATURE_COLS, _ = build_features(LANE)
y = LANE["severe_decline"].values
groups = LANE["client_id"].values

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(scores), kind="stable")
    top = order[:k] if k <= len(order) else order
    return float(np.mean(y_true[top])) if len(top) > 0 else 0.0

def run_one_regime(name, splitter, X_in, y_in, groups_in=None):
    fold_results = []
    for fi, (tr, te) in enumerate(splitter.split(X_in, y_in, groups_in)):
        X_tr, X_te = X_in[tr], X_in[te]
        y_tr, y_te = y_in[tr], y_in[te]
        # Strict check if regime is GROUPED: clients in train ∩ test must be empty
        if groups_in is not None:
            tr_cli = set(pd.unique(groups_in[tr]))
            te_cli = set(pd.unique(groups_in[te]))
            overlap = tr_cli & te_cli
            assert len(overlap) == 0, f"{name} fold {fi} client overlap={overlap}"
        rf = RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=5,
            n_jobs=1, random_state=RANDOM_STATE,
        )
        rf.fit(X_tr, y_tr)
        p = rf.predict_proba(X_te)[:, 1]
        p200 = precision_at_k(y_te, p, TOPK)
        p50  = precision_at_k(y_te, p, TOPK_SHORT)
        base_te = float(y_te.mean())
        fold_results.append({
            "fold_idx": fi, "n_train": len(tr), "n_test": len(te),
            "test_base_rate": base_te, f"prec@{TOPK}": p200, f"prec@{TOPK_SHORT}": p50,
        })
        print(f"  {name} fold {fi}:  test n={len(te):,}  base_rate={base_te:.1%}  "
              f"prec@{TOPK}={p200:.1%}  prec@{TOPK_SHORT}={p50:.1%}")
    return pd.DataFrame(fold_results)

# ===== REGIME 1: RANDOM SHUFFLE (naive) =====
print(f"\nREGIME 1 (BEFORE): ShuffleSplit random rows, {N_FOLDS} folds, test_size=0.2, random_state={RANDOM_STATE}")
ss = ShuffleSplit(n_splits=N_FOLDS, test_size=0.2, random_state=RANDOM_STATE)
res_random = run_one_regime("RANDOM", ss, X, y, groups_in=None)

# ===== REGIME 2: CLIENT GROUPED (honest) =====
print(f"\nREGIME 2 (AFTER): GroupShuffleSplit BY client_id, {N_FOLDS} folds, test_size=0.2, random_state={RANDOM_STATE}")
gss = GroupShuffleSplit(n_splits=N_FOLDS, test_size=0.2, random_state=RANDOM_STATE)
res_grouped = run_one_regime("GROUPED", gss, X, y, groups_in=groups)

# ===== BEFORE / AFTER COMPARISON TABLE =====
def agg(regime_df, name):
    return {
        "regime": name,
        "fold_rows": len(regime_df),
        f"prec@{TOPK} (mean±std)": f"{regime_df[f'prec@{TOPK}'].mean():.1%} ± "
                                 f"{(regime_df[f'prec@{TOPK}'].std(ddof=1) if N_FOLDS>1 else 0):.1%}",
        f"prec@{TOPK_SHORT} (mean±std)": f"{regime_df[f'prec@{TOPK_SHORT}'].mean():.1%} ± "
                                      f"{(regime_df[f'prec@{TOPK_SHORT}'].std(ddof=1) if N_FOLDS>1 else 0):.1%}",
        "mean_test_base_rate": f"{regime_df['test_base_rate'].mean():.1%}",
    }
base_row = {
    "regime": f"Base rate severe_decline ({BASE_RATE:.1%})",
    "fold_rows": N_FOLDS,
    f"prec@{TOPK} (mean±std)": f"{BASE_RATE:.1%}  (random)",
    f"prec@{TOPK_SHORT} (mean±std)": f"{BASE_RATE:.1%}  (random)",
    "mean_test_base_rate": f"{BASE_RATE:.1%}",
}
compare = pd.DataFrame([
    base_row,
    agg(res_random,  "Random ShuffleSplit (BEFORE, naive)"),
    agg(res_grouped, "GroupShuffleSplit by client (AFTER, honest)"),
])
gap_p200 = float(res_random[f"prec@{TOPK}"].mean() - res_grouped[f"prec@{TOPK}"].mean())
gap_p50  = float(res_random[f"prec@{TOPK_SHORT}"].mean() - res_grouped[f"prec@{TOPK_SHORT}"].mean())
print("\n" + "=" * 110)
print("BEFORE / AFTER COMPARISON TABLE (same model, same features, same metric, 5 folds)")
print("=" * 110)
print(compare.to_string(index=False))
print()
print(f"GAP (random score − honest grouped score):  prec@{TOPK} = {gap_p200:+.1%}  |  prec@{TOPK_SHORT} = {gap_p50:+.1%}")
print()
print("Interpretation (one sentence): ", end="")
if gap_p200 > +0.05:
    print(f"Random split inflates prec@{TOPK} by {gap_p200:+.1%} pp — the model was partially memorizing client-specific patterns it would not see on new customers. Honest grouped score is the only number we should forward-deploy with.")
elif gap_p200 < -0.02:
    print(f"Grouped score is actually HIGHER ({gap_p200:+.1%} gap negative) — test client groups happen to be easier; neither score is inflated, so both are credible.")
else:
    print(f"Small gap ({gap_p200:+.1%} pp) — very little client-memorization leakage in this RF; both scores are roughly credible.")
print()

# Save to metrics JSON
import json
metrics = {
    "label_base_rate": BASE_RATE,
    "regimes": {
        "random_split": res_random.to_dict(orient="records"),
        "grouped_split": res_grouped.to_dict(orient="records"),
    },
    "comparison_table": compare.to_dict(orient="records"),
    "gap_percentage_points": {"prec@200": float(gap_p200 * 100), "prec@50": float(gap_p50 * 100)},
}
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"Wrote validation audit receipt -> {METRICS_JSON_PATH}")

# Keep for S3 leakage audit
RANDOM_VS_GROUPED_GAP_P200 = float(gap_p200)
GROUPED_MEAN_P200 = float(res_grouped[f"prec@{TOPK}"].mean())
RANDOM_MEAN_P200  = float(res_random[f"prec@{TOPK}"].mean())



REGIME 1 (BEFORE): ShuffleSplit random rows, 5 folds, test_size=0.2, random_state=42


  RANDOM fold 0:  test n=4,402  base_rate=58.7%  prec@200=92.0%  prec@50=94.0%


  RANDOM fold 1:  test n=4,402  base_rate=60.4%  prec@200=89.0%  prec@50=92.0%


  RANDOM fold 2:  test n=4,402  base_rate=59.9%  prec@200=91.0%  prec@50=90.0%


  RANDOM fold 3:  test n=4,402  base_rate=60.6%  prec@200=88.0%  prec@50=92.0%


...

  GROUPED fold 0:  test n=3,614  base_rate=55.3%  prec@200=67.5%  prec@50=56.0%


  GROUPED fold 1:  test n=2,086  base_rate=78.5%  prec@200=87.5%  prec@50=80.0%


  GROUPED fold 2:  test n=3,029  base_rate=43.2%  prec@200=67.5%  prec@50=72.0%


  GROUPED fold 3:  test n=1,274  base_rate=47.8%  prec@200=78.0%  prec@50=82.0%


  GROUPED fold 4:  test n=1,484  base_rate=62.3%  prec@200=77.5%  prec@50=80.0%

BEFORE / AFTER COMPARISON TABLE (same model, same features, same metric, 5 folds)
                                     regime  fold_rows prec@200 (mean±std) prec@50 (mean±std) mean_test_base_rate
           Base rate severe_decline (59.7%)          5     59.7%  (random)    59.7%  (random)               59.7%
        Random ShuffleSplit (BEFORE, naive)          5        89.8% ± 1.6%       91.6% ± 1.7%               60.0%
GroupShuffleSplit by client (AFTER, honest)          5        75.6% ± 8.4%      74.0% ± 10.8%               57.4%

GAP (random score − honest grouped score):  prec@200 = +14.2%  |  prec@50 = +17.6%

Interpretation (one sentence): Random split inflates prec@200 by +14.2% pp — the model was partially memorizing client-specific patterns it would not see on new customers. Honest grouped score is the only number we should forward-deploy with.

Wrote validation audit receipt -> /Users/amiroyeleke

## 3. Leakage audit (attack my own model)

Now turn the full leakage-taxonomy attack on my Week-5 model (from the skill § attack checklist). Three taxons, three mechanical checks, then the "deliberately add a leak to confirm the harness works" exercise from the skill (train with it → score jumps → remove → score returns). Finally, the 9-point attack checklist, checked or unchecked with evidence.

**What a positive audit looks like:** One or two boxes unchecked with a concrete reason = we understand the boundary; every box checked with a mechanical proof = honest. If the deliberately-added leak doesn't jump the score toward 1.0, the harness is broken — and that finding is itself more honest than any number.


In [3]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

print("=" * 110)
print("3a. LEAKAGE TAXON 1: Label-derived feature audit (train-with vs train-without test)")
print("=" * 110)
print("\nKnown label-source columns: columns that trend_pct label was COMPUTED FROM or are siblings.")
print("Suspects per lane guide: trend_pct, trend_direction, is_declining, decline_severity, opportunity_score.")
known_label_cols = {"trend_pct", "trend_direction", "is_declining", "decline_severity",
                    "opportunity_score", "severe_decline"}
lane_cols = set(LANE.columns)
suspects_present = sorted(known_label_cols & lane_cols)
print(f"  Present in LANE frame (possible to accidentally include): {suspects_present}")
print(f"  Honest FEATURE_COLS used by model: {FEATURE_COLS}")
print(f"  Feature cols ∩ known label cols: {sorted(set(FEATURE_COLS) & known_label_cols) or 'CLEAN (empty)'}")

# Mechanical proof: train ONCE with trend_pct deliberately added, once without, show score jump
# (from the skill § How to verify: deliberately ADD a leak — if doesn't jump → harness broken)
# Same grouped regime fold 0 (single fold, quick)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr, te = next(iter(gss.split(X, y, groups)))
X_honest_tr, X_honest_te = X[tr], X[te]
y_tr, y_te = y[tr], y[te]
# Build LEAKED version: append trend_pct as a column
trend_pct_vals = LANE["trend_pct"].fillna(0).astype(float).values.reshape(-1, 1)
X_leaked_tr = np.hstack([X_honest_tr, trend_pct_vals[tr]])
X_leaked_te = np.hstack([X_honest_te, trend_pct_vals[te]])
rf_honest = RandomForestClassifier(200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1)
rf_leaked = RandomForestClassifier(200, max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1)
rf_honest.fit(X_honest_tr, y_tr); honest_p200 = precision_at_k(y_te, rf_honest.predict_proba(X_honest_te)[:,1], TOPK)
rf_leaked.fit(X_leaked_tr, y_tr);  leaked_p200  = precision_at_k(y_te, rf_leaked.predict_proba(X_leaked_te)[:,1],  TOPK)
print(f"  PROOF:  honest prec@{TOPK} (label columns NOT in X) = {honest_p200:.1%}")
print(f"  PROOF:  leaked prec@{TOPK} (trend_pct ADDED to X)    = {leaked_p200:.1%}")
confession = (leaked_p200 - honest_p200) > 0.08  # jumps >8 pp = harness catches it
print(f"  Score jump = {leaked_p200-honest_p200:+.1%}. Harness catches it? {'YES (good)' if confession else 'NO (harness BROKEN)'}")
print("  (After this mechanical proof, we discard the LEAKED model — honest number stays forward.)")
print()

print("=" * 110)
print("3b. LEAKAGE TAXON 2: Future / overlapping windows — timeline drawn")
print("=" * 110)
print("""
  Timing contract (per w03_data_contract + starter export description):
    [FEATURE WINDOW = 90 trailing days before snapshot date]
        impressions_90d, ctr, avg_position, search_volume (all GSC trailing-90d aggregates)
        content_age_days (snapshot_date − publish_date, static property)
        days_since_last_update (snapshot_date − last_update, static property)
        staleness/vis/striking_bucket (derived FROM the above only)
    [SNAPSHOT / DECISION MOMENT]
    [LABEL WINDOW = trend_pct is the observed outcome column, end-to-end of the 90-day window]
        Label is severe_decline = trend_pct < -20.
  Does any feature aggregate over data that touches or exceeds the decision moment?
    — On the starter export, every feature column is explicitly a trailing-90-day snapshot aggregate:
      each feature caption says "90d" and the column was pulled from the snapshot export description.
    — On the warehouse (per w03 contract): every feature column must be `WHERE report_date < label_month_start`.
      We do not have the warehouse panel here, but we carry forward that known limitation.
""")
# Mechanical: check that for every feature column, no feature range or description mentions forward/label term
terms_forward = ["forward", "label month", "trend", "decline_severity", "opportunity"]
# For starter export: verify that features are aggregates of trailing columns only
features_source = {
    "log_impressions_90d": "impressions_90d (90-day trailing imp)",
    "ctr_filled": "ctr (GSC 90-day trailing CTR)",
    "position_filled": "avg_position (GSC 90-day trailing position)",
    "log_search_volume": "search_volume (keyword volume, static at snapshot)",
    "age_days": "content_age_days (snap - publish, static)",
    "days_since_update": "days_since_last_update (snap - last_update, static)",
    "staleness_bucket": "derived from days_since_last_update (snapshot metadata)",
    "vis_bucket": "derived from impressions_90d (trailing)",
    "striking_bonus": "derived from avg_position + search_volume (both trailing / static)",
    "has_word_count": "word_count existence flag, content metadata",
    "word_count_filled": "word_count value, content metadata",
}
print("  Feature × timeline audit:")
for f, source in features_source.items():
    bad = [t for t in terms_forward if t in source.lower()]
    print(f"    {f:25s} <- {source}  |  forward-term: {'OK' if not bad else f'FLAG: {bad}'}")
print()

print("=" * 110)
print("3c. LEAKAGE TAXON 3: Decision-derived features (existing FlyRank flags / product rules)")
print("=" * 110)
print("  Flagged columns per the skill: refresh_flag_stale, ctr_flag, quick_win_flag, or any existing action-score.")
flag_cols = [c for c in lane_cols if any(t in c.lower() for t in ["flag", "action", "score", "refresh"])]
print(f"  Flag / existing-score columns present in LANE frame: {flag_cols or '(none found — clean export)'}")
print(f"  Honest FEATURE_COLS: {FEATURE_COLS}")
print(f"  Feature ∩ flag columns: {sorted(set(FEATURE_COLS) & set(flag_cols)) or 'CLEAN (no product flags in features)'}")
if flag_cols:
    print("  (Note: staleness_bucket is MY OWN derived bucket, not a product flag — it uses raw days_since_last_update only.)")
print()

print("=" * 110)
print("3d. THE 9-POINT ATTACK CHECKLIST (from the skill, checked with mechanical proof)")
print("=" * 110)
checklist = {
    "1. Timeline drawn: all features strictly before label window":
        ("YES (§3b audit: every feature is 90d-trailing snapshot or static metadata)", True),
    "2. No label-derived or sibling columns in features":
        (f"YES (§3a: train-with trend_pct = {leaked_p200:.0%} vs train-without = {honest_p200:.0%}. "
         f"Honest frame ∩ label cols = empty)", True),
    "3. No product flags / existing-system scores as features":
        (f"YES (§3c: no flag/action/refresh columns in features — clean export)", True),
    "4. Population selection checked for outcome-window info":
        ("LIMITATION DISCLOSED: starter export is a one-time 90d snapshot. "
         "Lane filter (imp_90d ≥ 100) is trailing-90d (OK), but we cannot "
         "fully verify 'still-active-in-label-month' clients without warehouse panel. "
         "Per skill this is a CHOICE, hidden only if undisclosed — we disclose it here.", False),
    "5. Split grouped by repeating entity (client_id)":
        (f"YES (§2: GroupShuffleSplit by client_id; per-fold overlap asserted empty)", True),
    "6. Base rate printed next to every metric":
        (f"YES (base rate = {BASE_RATE:.1%}; printed in §2 next to both prec@200 and prec@50)", True),
    "7. Top feature importance sanity-checked (no giant gap)":
        (f"LIMITATION DISCLOSED: permutation-importance top feature was CTR at 11.5 pp (§4a w05), "
         f"not a 60 pp 'perfect' gap — OK, no leakage red flag", True),
    "8. Metrics recomputed out-of-fold (never in-sample)":
        ("YES (§2 folds use held-out test only for scoring; no in-sample train score reported)", True),
    "9. Sealed/holdout claims: receipts committed":
        ("Receipt file validation_audit.json written to work/outputs/ "
         "(§2 comparison table + §3 audit). Committed with this notebook.", True),
}
for i, (item, (explanation, passed)) in enumerate(checklist.items(), 1):
    mark = "[x]" if passed else "[~]"
    print(f"  {mark} {item}")
    print(f"      -> {explanation}")
print()

# Save checklist to metrics JSON
import json
with open(METRICS_JSON_PATH, "r") as f:
    m = json.load(f)
m["leakage_audit"] = {
    "taxon_1_label_derived": {
        "honest_prec200": float(honest_p200),
        "leaked_prec200": float(leaked_p200),
        "feature_suspect_intersection_empty": True,
        "harness_confession_jump_percentage_points": float((leaked_p200 - honest_p200) * 100),
    },
    "taxon_2_overlapping_windows": {
        "timeline_drawn": True,
        "disclosed_limitation": "Starter export only; warehouse must enforce report_date < label_month_start.",
    },
    "taxon_3_decision_flags": {
        "flag_intersection_empty": len(set(FEATURE_COLS) & set(flag_cols)) == 0,
    },
    "attack_checklist_9_point": [
        {"item": k, "explanation": v[0], "passed": bool(v[1])} for k, v in checklist.items()
    ],
    "unverified_limitation_disclosed": "Checklist #4 not 100% verifiable on starter export; warehouse population must re-check 'active-in-label-month' filter independence.",
}
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(m, f, indent=2, default=str)
print(f"Leakage audit updated in metrics receipt -> {METRICS_JSON_PATH}")


3a. LEAKAGE TAXON 1: Label-derived feature audit (train-with vs train-without test)

Known label-source columns: columns that trend_pct label was COMPUTED FROM or are siblings.
Suspects per lane guide: trend_pct, trend_direction, is_declining, decline_severity, opportunity_score.
  Present in LANE frame (possible to accidentally include): ['severe_decline', 'trend_direction', 'trend_pct']
  Honest FEATURE_COLS used by model: ['log_impressions_90d', 'ctr_filled', 'position_filled', 'log_search_volume', 'age_days', 'days_since_update', 'staleness_bucket', 'vis_bucket', 'striking_bonus', 'has_word_count', 'word_count_filled']
  Feature cols ∩ known label cols: CLEAN (empty)


  PROOF:  honest prec@200 (label columns NOT in X) = 67.5%
  PROOF:  leaked prec@200 (trend_pct ADDED to X)    = 100.0%
  Score jump = +32.5%. Harness catches it? YES (good)
  (After this mechanical proof, we discard the LEAKED model — honest number stays forward.)

3b. LEAKAGE TAXON 2: Future / overlapping windows — timeline drawn

  Timing contract (per w03_data_contract + starter export description):
    [FEATURE WINDOW = 90 trailing days before snapshot date]
        impressions_90d, ctr, avg_position, search_volume (all GSC trailing-90d aggregates)
        content_age_days (snapshot_date − publish_date, static property)
        days_since_last_update (snapshot_date − last_update, static property)
        staleness/vis/striking_bucket (derived FROM the above only)
    [SNAPSHOT / DECISION MOMENT]
    [LABEL WINDOW = trend_pct is the observed outcome column, end-to-end of the 90-day window]
        Label is severe_decline = trend_pct < -20.
  Does any feature aggregate over data tha

## 4. Claim rewrite (safe language vs over-stretched)

This audit's final test: the honest score is only honest if the words around it are honest too. Take 3 confident-sounding phrases from my Week-5 model summary and rewrite each one using only "observed / measured / directional / decision-support" language that a reviewer could falsify with code.

The rewrite rule:
- Replace **"Our model beats the baseline"** with **"On this 22,006-row starter-export slice, measured on 5 client-held-out folds, the Random Forest's mean precision@200 was +22.7 pp higher than the frozen Week-4 rule's mean precision@200 (75.6% vs 52.9%)"**.
- Replace claims of causality ("X causes Y") with directional claims ("X was directionally associated with Y in measured OOF predictions").
- Replace "generalization" claims with "on this split, on this data export" qualifiers.


In [4]:
print("=" * 110)
print("CLAIM #1 — Week-5 strong-sounding ORIGINAL → safe rewrite")
print("=" * 110)
orig1 = """ORIGINAL (over-stretched — the kind of line I almost wrote in Week-5):
  "Random Forest achieves excellent precision and generalizes well to new clients."
"""
rew1 = f"""REWRITE (measured + decision-support + scoped):
  "On this 22,006-row starter-export Lane 2 slice, measured across 5 client-held-out
   GroupShuffleSplit folds (test_size=0.2, random_state={RANDOM_STATE}), the Random Forest
   (200 trees, max_depth=6, min_samples_leaf=5) produced a mean prec@{TOPK} of
   {GROUPED_MEAN_P200*100:.1f}%. On the same folds, the frozen Week-4 rule scored 52.9%;
   the naive random-shuffle baseline (§2 BEFORE) scored {RANDOM_MEAN_P200*100:.1f}%,
   a gap of {RANDOM_VS_GROUPED_GAP_P200*100:+.1f} pp, indicating the model partially
   memorizes client-specific patterns on a random split. The model is a decision-support
   tool for ranking an editor queue. 'Generalizes well to brand-new FlyRank customers' is
   a claim this starter-export notebook does not have evidence for — deployment evidence
   would need a forward-month time-based holdout on the warehouse panel."
"""
print(orig1)
print(rew1)

print("=" * 110)
print("CLAIM #2 — Week-5 strong-sounding ORIGINAL → safe rewrite")
print("=" * 110)
orig2 = """ORIGINAL (over-stretched, causal language):
  "CTR is the #1 driver of content decline, and staleness causes SEO slip."
"""
rew2 = f"""REWRITE (directional + measured + non-causal):
  "On the last-fold client-held-out prec@{TOPK} score, CTR-filled (0 where missing) was
   the feature with the largest observed permutation-importance drop (11.5 pp ± 2.4 pp
   drop when shuffled — w05 §4a). Content age ranked second at 3.9 pp drop; position
   third at 1.7 pp. In train-fold correlation, CTR was directionally associated with
   severe_decline (corr ≈ −0.08: lower CTR → more severe decline was OBSERVED in this
   export). These are measured, directional associations on the 22,006-row starter
   snapshot — not causal claims. A causal claim would need an A/B rollout where CTR
   is changed and SEO slip measured.
"""
print(orig2)
print(rew2)

print("=" * 110)
print("CLAIM #3 — Week-5 strong-sounding ORIGINAL → safe rewrite")
print("=" * 110)
orig3 = """ORIGINAL (over-stretched, absolute-quality claim):
  "This model is ready for production; it finds more declining pages and is much better than the rule."
"""
rew3 = f"""REWRITE (scoped + population + with honest error caveat):
  "Measured across 5 OOF client-held-out folds on the starter export, the RF had a
   +22.7 pp higher mean prec@{TOPK} than the frozen Week-4 rule (75.6% vs 52.9%).
   The same score on a naive random split was {RANDOM_MEAN_P200*100:.1f}%, so
   {RANDOM_VS_GROUPED_GAP_P200*100:+.1f} pp of that 'big win' is client-memorization
   the honest grouped split correctly removes. Out-of-fold error analysis on the pooled
   5 folds (w05 §4b) shows 244 top-200 false positives (wasted editor slots) and 5,720
   false negatives (hidden opportunities the model still misses). This model is a
   decision-support ranking improvement ON THIS EXPORT. 'Ready for production' would
   need: (1) a sealed forward-month time-based holdout on the warehouse panel,
   (2) re-verification that the grouped-split gain holds on new months, and
   (3) the disclosed known limitation addressed: last-30-day shape signals are
   currently invisible to this 90-day-snapshot feature set."
"""
print(orig3)
print(rew3)
print("=" * 110)
print("Safe-language vocabulary legend (use these everywhere):")
legend = pd.DataFrame({
    "Avoid (strong / absolute)": [
        "beats / excels at",
        "generalizes well to new clients",
        "drives / causes",
        "ready for production",
        "finds all / fixes",
    ],
    "Use instead (measured / directional / decision-support)": [
        "was measured to be X pp higher on Y folds than Z",
        f"shows a {RANDOM_VS_GROUPED_GAP_P200*100:+.1f} pp gap between random and grouped (§2)",
        "was directionally associated with (corr = +/− X) in this export",
        "is a decision-support ranking improvement on this export; deployment needs sealed forward holdout",
        "observes fewer false positives / false negatives ON THIS OOF DATA",
    ],
})
print(legend.to_string(index=False))
print()

# Save claim rewrites to JSON receipt
import json
with open(METRICS_JSON_PATH, "r") as f:
    m = json.load(f)
m["claim_rewrites"] = [
    {"id": 1, "original_strong": orig1, "rewritten_safe": rew1},
    {"id": 2, "original_strong": orig2, "rewritten_safe": rew2},
    {"id": 3, "original_strong": orig3, "rewritten_safe": rew3},
    {"vocabulary_legend": legend.to_dict(orient="records")},
]
with open(METRICS_JSON_PATH, "w") as f:
    json.dump(m, f, indent=2, default=str)
print(f"Claim rewrites appended to receipt -> {METRICS_JSON_PATH}")


CLAIM #1 — Week-5 strong-sounding ORIGINAL → safe rewrite
ORIGINAL (over-stretched — the kind of line I almost wrote in Week-5):
  "Random Forest achieves excellent precision and generalizes well to new clients."

REWRITE (measured + decision-support + scoped):
  "On this 22,006-row starter-export Lane 2 slice, measured across 5 client-held-out
   GroupShuffleSplit folds (test_size=0.2, random_state=42), the Random Forest
   (200 trees, max_depth=6, min_samples_leaf=5) produced a mean prec@200 of
   75.6%. On the same folds, the frozen Week-4 rule scored 52.9%;
   the naive random-shuffle baseline (§2 BEFORE) scored 89.8%,
   a gap of +14.2 pp, indicating the model partially
   memorizes client-specific patterns on a random split. The model is a decision-support
   tool for ranking an editor queue. 'Generalizes well to brand-new FlyRank customers' is
   a claim this starter-export notebook does not have evidence for — deployment evidence
   would need a forward-month time-based holdout

## Self-check

Before you submit, confirm each line honestly:

- [x] §1 contains two paper findings and a concrete, constructive methodology question for each — the exact same questions I would want a reviewer to ask me.
- [x] §2 reruns the Week-5 model under a BEFORE (random ShuffleSplit) vs AFTER (client GroupShuffleSplit) regime and reports both numbers + the GAP. The honest grouped score is the one forwarded.
- [x] §3 runs the full 3-taxonomy leakage attack on my own model: label-derived features (with the deliberate-leak confession test), timeline windows, product-flag features. All 9 attack-checklist boxes are checked or have a disclosed limitation.
- [x] §4 rewrites three strong-sounding Week-5 claims into scoped "observed / measured / directional / decision-support" language. No absolute-quality or causal-sounding lines survive.
- [x] The notebook runs top to bottom with no errors (nbconvert executed 5 code cells, 0 exceptions). Validation receipts committed to `work/outputs/validation_audit.json`.
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
